In [0]:
CREATE OR REPLACE TABLE fuel_price_dev.gold.agg_taux_rupture_par_region AS
WITH stations_par_region AS (
    SELECT
        g.code_region,
        COUNT(DISTINCT st.station_id) AS nb_stations_total
    FROM fuel_price_dev.silver.dim_station st
    LEFT JOIN fuel_price_dev.silver.dim_geo g
        ON st.code_departement = g.code_departement
    GROUP BY g.code_region
),
ruptures_par_region_jour AS (
    SELECT
        CAST(rpt.debut_rupture AS DATE) AS date,
        g.code_region,
        car.nom AS carburant,
        COUNT(DISTINCT rpt.station_id) AS nb_stations_rupture
    FROM fuel_price_dev.silver.fait_rupture rpt
    LEFT JOIN fuel_price_dev.silver.dim_carburant car
        ON rpt.id_carburant = car.id
    LEFT JOIN fuel_price_dev.silver.dim_station st
        ON st.station_id = rpt.station_id
    LEFT JOIN fuel_price_dev.silver.dim_geo g
        ON st.code_departement = g.code_departement
    WHERE rpt.fin_rupture IS NULL  
    GROUP BY CAST(rpt.debut_rupture AS DATE), g.code_region, car.nom
)
SELECT
    r.date,
    r.code_region,
    r.carburant,
    s.nb_stations_total,
    r.nb_stations_rupture,
    ROUND(r.nb_stations_rupture / NULLIF(s.nb_stations_total, 0), 3) AS taux_rupture
FROM ruptures_par_region_jour r
LEFT JOIN stations_par_region s
    ON r.code_region = s.code_region
ORDER BY r.date, r.code_region, r.carburant;